# 02 — Draw a Bounding Box

You can now compute a bbox as four numbers. This notebook turns those numbers into a visible rectangle on a map.

The goal is simple: overlay the bbox on top of the original data and confirm that the summary actually covers what you think it covers. Numbers become geometry. Geometry becomes a shape you can look at.

## 1. Review: What BBox Values Mean

A bbox is `[min_lon, min_lat, max_lon, max_lat]`. Each value is a side of a rectangle:

```
[min_lon,  min_lat,  max_lon,  max_lat]
  west      south     east      north
```

To draw it, you need to convert those four edge values into **four corner coordinates** — then connect them into a closed ring.

## 2. Convert BBox to Corner Coordinates

The four corners of the rectangle are combinations of the min/max values:

| Corner    | Longitude  | Latitude  |
|-----------|------------|-----------|
| Southwest | `min_lon`  | `min_lat` |
| Southeast | `max_lon`  | `min_lat` |
| Northeast | `max_lon`  | `max_lat` |
| Northwest | `min_lon`  | `max_lat` |

**Important:** GeoJSON stores coordinates as `[lon, lat]`. The `ipyleaflet` `Map` and `Marker` classes expect `(lat, lon)`. The `GeoJSON` layer class reads GeoJSON directly, so its coordinates stay in `[lon, lat]` order.

That inconsistency deserves public shaming — but it exists, so plan accordingly.

In [1]:
# Start with a bbox computed from the previous notebook
bbox = [-99.0, 32.9, -97.2, 34.6]
min_lon, min_lat, max_lon, max_lat = bbox

# Four corners in [lon, lat] order (GeoJSON convention)
sw = [min_lon, min_lat]
se = [max_lon, min_lat]
ne = [max_lon, max_lat]
nw = [min_lon, max_lat]

print("Southwest:", sw)
print("Southeast:", se)
print("Northeast:", ne)
print("Northwest:", nw)

Southwest: [-99.0, 32.9]
Southeast: [-97.2, 32.9]
Northeast: [-97.2, 34.6]
Northwest: [-99.0, 34.6]


## 3. Build the BBox as a GeoJSON Polygon

A GeoJSON Polygon needs a **closed ring** — a list of coordinate pairs where the first and last pair are identical. For a rectangle, that means five entries: the four corners plus a repeat of the first corner.

Walk the ring in order: SW → SE → NE → NW → SW.

In [2]:
def bbox_to_polygon(bbox):
    """
    Convert [min_lon, min_lat, max_lon, max_lat] into a GeoJSON Polygon Feature.
    """
    min_lon, min_lat, max_lon, max_lat = bbox

    ring = [
        [min_lon, min_lat],   # SW
        [max_lon, min_lat],   # SE
        [max_lon, max_lat],   # NE
        [min_lon, max_lat],   # NW
        [min_lon, min_lat],   # SW — closes the ring
    ]

    return {
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [ring]
        },
        "properties": {}
    }


bbox_feature = bbox_to_polygon(bbox)
print("Ring vertices:")
for coord in bbox_feature["geometry"]["coordinates"][0]:
    print(" ", coord)

Ring vertices:
  [-99.0, 32.9]
  [-97.2, 32.9]
  [-97.2, 34.6]
  [-99.0, 34.6]
  [-99.0, 32.9]


## 4. Draw the BBox on a Map

Add the original data and the bbox rectangle as two separate `GeoJSON` layers. The `GeoJSON` layer accepts a FeatureCollection dict and renders everything inside it — coordinates stay in `[lon, lat]` order.

The map center uses `(lat, lon)` order — ipyleaflet's convention.

In [3]:
from ipyleaflet import Map, GeoJSON

# Original point data
points_fc = {
    "type": "FeatureCollection",
    "features": [
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.5, 33.8]}, "properties": {"name": "A"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.2, 34.1]}, "properties": {"name": "B"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-99.0, 32.9]}, "properties": {"name": "C"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-98.1, 33.4]}, "properties": {"name": "D"}},
        {"type": "Feature", "geometry": {"type": "Point", "coordinates": [-97.8, 34.6]}, "properties": {"name": "E"}},
    ]
}

# Compute and build bbox
bbox = [-99.0, 32.9, -97.2, 34.6]
bbox_fc = {
    "type": "FeatureCollection",
    "features": [bbox_to_polygon(bbox)]
}

# Map center: average of bbox midpoint — ipyleaflet wants (lat, lon)
center_lat = (bbox[1] + bbox[3]) / 2
center_lon = (bbox[0] + bbox[2]) / 2

m = Map(center=(center_lat, center_lon), zoom=8)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=bbox_fc))
m

Map(center=[33.75, -98.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

## 5. Style the BBox

The default polygon style makes the bbox look like a filled feature competing with the data. Use a transparent fill and a distinct stroke color so it reads clearly as a boundary rather than a shape.

In [4]:
bbox_style = {
    "color": "#e63946",       # stroke color — red-orange
    "weight": 2,              # stroke width in pixels
    "fillColor": "#e63946",
    "fillOpacity": 0.05,      # nearly transparent fill
}

m = Map(center=(center_lat, center_lon), zoom=8)
m.add(GeoJSON(data=points_fc))
m.add(GeoJSON(data=bbox_fc, style=bbox_style))
m

Map(center=[33.75, -98.1], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

## 6. BBox vs. Original Feature Geometry

The bbox of a polygon is almost always larger than the polygon itself — it is the smallest rectangle that contains the shape, not the shape. Irregular or diagonal geometry produces especially loose bounding rectangles.

Look at the map below and notice:
- the bbox contains the polygon completely
- the corners of the bbox are empty space
- the tighter the original shape, the better the bbox approximates it

In [5]:
# An irregular polygon — not a rectangle
irregular_fc = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [-98.8, 33.2],
                [-97.5, 33.0],
                [-97.1, 33.9],
                [-97.8, 34.5],
                [-98.7, 34.2],
                [-99.0, 33.6],
                [-98.8, 33.2],
            ]]
        },
        "properties": {"name": "Irregular Shape"}
    }]
}

# Compute bbox from the polygon's exterior ring
ring = irregular_fc["features"][0]["geometry"]["coordinates"][0]
lons = [v[0] for v in ring]
lats = [v[1] for v in ring]
poly_bbox = [min(lons), min(lats), max(lons), max(lats)]

poly_bbox_fc = {
    "type": "FeatureCollection",
    "features": [bbox_to_polygon(poly_bbox)]
}

poly_style  = {"color": "#457b9d", "weight": 2, "fillOpacity": 0.15}
bbox_style2 = {"color": "#e63946", "weight": 2, "fillOpacity": 0.05}

center_lat2 = (poly_bbox[1] + poly_bbox[3]) / 2
center_lon2 = (poly_bbox[0] + poly_bbox[2]) / 2

m = Map(center=(center_lat2, center_lon2), zoom=8)
m.add(GeoJSON(data=irregular_fc,  style=poly_style))
m.add(GeoJSON(data=poly_bbox_fc,  style=bbox_style2))
m

Map(center=[33.75, -98.05], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## 7. Zoom the Map to the BBox

Rather than guessing a center and zoom level, you can fit the map view to the bbox directly. ipyleaflet's `Map` accepts a `bounds` parameter as `[[south, west], [north, east]]` — note the `(lat, lon)` order again.

In [6]:
def bbox_to_map_bounds(bbox):
    """
    Convert [min_lon, min_lat, max_lon, max_lat] to ipyleaflet bounds:
    [[south, west], [north, east]]
    """
    min_lon, min_lat, max_lon, max_lat = bbox
    return [[min_lat, min_lon], [max_lat, max_lon]]


bounds = bbox_to_map_bounds(poly_bbox)
print("ipyleaflet bounds:", bounds)

# Fit the map to the bbox instead of setting center + zoom manually
m = Map()
m.fit_bounds(bounds)
m.add(GeoJSON(data=irregular_fc, style=poly_style))
m.add(GeoJSON(data=poly_bbox_fc, style=bbox_style2))
m

ipyleaflet bounds: [[33.0, -99.0], [34.5, -97.1]]


Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

## 8. BBox as a Quick Filter

A bbox is not just for visualization. It is a fast pre-test: if a point is outside the bbox, it cannot be inside the original geometry — so you can skip it without doing any expensive computation.

This is the idea behind spatial indexing: narrow a large candidate set down quickly using cheap rectangle checks, then apply precise tests only to the survivors.

Common uses:
- **"Maybe inside"** — if a point is inside the bbox, it *might* be inside the polygon. If it is outside the bbox, it definitely is not
- **View window** — determine which features are even visible at the current map zoom
- **Rough collision check** — do two features overlap at all? Check their bboxes first
- **Narrowing search** — filter 100,000 records to 200 bbox candidates before running the real query

You do not need any of this machinery right now. Just understand that the rectangle you drew has utility beyond looking neat.

---

## Exercise A — Compute and Draw from Point Data

Given the airfield coordinates below, compute the bbox and draw it on a map alongside the original points. Use a styled bbox layer that is clearly distinct from the point markers.

In [9]:
airfields_fc = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [-97.37, 35.39]
            },
            "properties": {
                "name": "Tinker AFB"
            }
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [-94.37, 35.34]
            },
            "properties": {
                "name": "Fort Smith Regional"
            }
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [-101.70, 33.66]
            },
            "properties": {
                "name": "Lubbock Preston Smith"
            }
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [-97.04, 32.85]
            },
            "properties": {
                "name": "NAS Fort Worth JRB"
            }
        },
        {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [-98.47, 29.53]
            },
            "properties": {
                "name": "Kelly Field Annex"
            }
        }
    ]
}

In [10]:
from ipyleaflet import Map, GeoJSON, Marker
from ipywidgets import HTML

def compute_bbox(coords):
    lons = []
    lats = []

    for lon, lat in coords:
        lons.append(lon)
        lats.append(lat)

    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_to_polygon(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox

    return {
        "type": "Polygon",
        "coordinates": [[
            [min_lon, min_lat],
            [max_lon, min_lat],
            [max_lon, max_lat],
            [min_lon, max_lat],
            [min_lon, min_lat]
        ]]
    }


# 1. extract coordinates
coords = []

for feature in airfields_fc["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    coords.append([lon, lat])


# 2. compute bbox
bbox = compute_bbox(coords)
print("BBox:", bbox)


# 3. build bbox polygon
bbox_polygon = bbox_to_polygon(bbox)

bbox_feature = {
    "type": "Feature",
    "geometry": bbox_polygon,
    "properties": {}
}


# 4. display map
m = Map(center=(33.5, -98.0), zoom=5)

# add original point markers
for feature in airfields_fc["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    name = feature["properties"]["name"]

    marker = Marker(
        location=(lat, lon),
        draggable=False,
        title=name
    )

    marker.popup = HTML(value=name)
    m.add(marker)


# add styled bbox layer
bbox_layer = GeoJSON(
    data=bbox_feature,
    style={
        "color": "red",
        "weight": 3,
        "fillColor": "red",
        "fillOpacity": 0.1
    }
)

m.add(bbox_layer)

# fit map to bbox
min_lon, min_lat, max_lon, max_lat = bbox
m.fit_bounds([
    [min_lat, min_lon],
    [max_lat, max_lon]
])

m

BBox: [-101.7, 29.53, -94.37, 35.39]


Map(center=[33.5, -98.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Exercise B — Overlay Polygon and Its BBox

Draw the irregular polygon from Section 6 and its bbox on the same map. Use different colors for each. How much empty space is inside the bbox but outside the polygon?

In [11]:
from ipyleaflet import Map, GeoJSON

# Make the bbox polygon from the bbox
bbox_polygon = bbox_to_polygon(poly_bbox)

bbox_feature = {
    "type": "Feature",
    "geometry": bbox_polygon,
    "properties": {}
}

# Get bbox values
min_lon, min_lat, max_lon, max_lat = poly_bbox

# Create map
m = Map(center=((min_lat + max_lat) / 2, (min_lon + max_lon) / 2), zoom=6)

# Original irregular polygon layer
polygon_layer = GeoJSON(
    data=irregular_fc,
    style={
        "color": "blue",
        "weight": 3,
        "fillColor": "blue",
        "fillOpacity": 0.3
    }
)

# BBox layer
bbox_layer = GeoJSON(
    data=bbox_feature,
    style={
        "color": "red",
        "weight": 3,
        "fillColor": "red",
        "fillOpacity": 0.1
    }
)

m.add(polygon_layer)
m.add(bbox_layer)

m.fit_bounds([
    [min_lat, min_lon],
    [max_lat, max_lon]
])

m

Map(center=[33.75, -98.05], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_…

## Exercise C — Multi-Feature BBox Layers

For each airfield in `airfields_fc`, draw its individual bbox (a degenerate rectangle for a point — same coordinates on all four corners) using a distinct color per feature. All five bboxes should appear on a single map alongside the points.

In [12]:
from ipyleaflet import Map, GeoJSON, Marker
from ipywidgets import HTML

colors = ["#e63946", "#2a9d8f", "#e9c46a", "#f4a261", "#264653"]

m = Map(center=(33.5, -98.0), zoom=5)

for i, feature in enumerate(airfields_fc["features"]):
    lon, lat = feature["geometry"]["coordinates"]
    name = feature["properties"]["name"]

    # add point marker
    marker = Marker(
        location=(lat, lon),
        draggable=False,
        title=name
    )
    marker.popup = HTML(value=name)
    m.add(marker)

    # compute individual bbox for this point
    bbox = compute_bbox([[lon, lat]])

    # build bbox polygon
    bbox_polygon = bbox_to_polygon(bbox)

    bbox_feature = {
        "type": "Feature",
        "geometry": bbox_polygon,
        "properties": {"name": name}
    }

    # add styled bbox layer
    bbox_layer = GeoJSON(
        data=bbox_feature,
        style={
            "color": colors[i],
            "weight": 4,
            "fillColor": colors[i],
            "fillOpacity": 0.3
        }
    )

    m.add(bbox_layer)

# fit map to full airfields bbox
all_coords = []

for feature in airfields_fc["features"]:
    all_coords.append(feature["geometry"]["coordinates"])

full_bbox = compute_bbox(all_coords)
min_lon, min_lat, max_lon, max_lat = full_bbox

m.fit_bounds([
    [min_lat, min_lon],
    [max_lat, max_lon]
])

m

Map(center=[33.5, -98.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

## Exercise D — Fit the Map to the Data

Using `airfields_fc` and its bbox, display the map using `fit_bounds` rather than a hardcoded center and zoom. Confirm that all points are visible without manual adjustment.

In [13]:
from ipyleaflet import Map, Marker
from ipywidgets import HTML

def compute_bbox(coords):
    lons = []
    lats = []

    for lon, lat in coords:
        lons.append(lon)
        lats.append(lat)

    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_to_map_bounds(bbox):
    min_lon, min_lat, max_lon, max_lat = bbox

    return [
        [min_lat, min_lon],
        [max_lat, max_lon]
    ]


# extract coordinates
coords = []

for feature in airfields_fc["features"]:
    coords.append(feature["geometry"]["coordinates"])


# compute bbox
bbox = compute_bbox(coords)

# create map
m = Map()

# add airfield markers
for feature in airfields_fc["features"]:
    lon, lat = feature["geometry"]["coordinates"]
    name = feature["properties"]["name"]

    marker = Marker(
        location=(lat, lon),
        draggable=False,
        title=name
    )

    marker.popup = HTML(value=name)
    m.add(marker)


# fit map to data
bounds = bbox_to_map_bounds(bbox)
m.fit_bounds(bounds)

m

Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

---

## Check Your Understanding

You write the following to draw a bbox and the map renders — but the rectangle appears in the wrong location, off in the Atlantic Ocean.

```python
bbox = [-99.0, 32.9, -97.2, 34.6]
min_lon, min_lat, max_lon, max_lat = bbox

ring = [
    [min_lat, min_lon],
    [min_lat, max_lon],
    [max_lat, max_lon],
    [max_lat, min_lon],
    [min_lat, min_lon],
]
```

**Question:** What is wrong with this code? What specific value ends up as the longitude in the first vertex, and why does that send the rectangle to the wrong place?

```python
# your answer here
```


---

In [14]:
# The problem is that the coordinates were written in [latitude, longitude] order instead of GeoJSON's required [longitude, latitude] order.

# The code incorrectly does:
[min_lat, min_lon]

# which becomes:
[32.9, -99.0]

# GeoJSON interprets that as:
# longitude = 32.9
# latitude = -99.0

# But latitude cannot be less than -90, and longitude 32.9 places the shape near Europe/Africa instead of the United States.

# The rectangle is sent to the wrong place because latitude and longitude were swapped.

# The correct ring should be:

ring = [
    [min_lon, min_lat],
    [max_lon, min_lat],
    [max_lon, max_lat],
    [min_lon, max_lat],
    [min_lon, min_lat],
]

## Next

In [03 — Why Lat/Lon Is Weird](./03-Why_LatLon_Is_Weird.ipynb), we confront why degrees are not meters and when your flat-Earth shortcuts quietly produce wrong answers.